# Medical Coding AI: pre-bill coding & claim-integrity assistant (Colab)

**What it does:** clinical note -> ICD-10-CM diagnoses + CPT/HCPCS (outpatient) or ICD-10-PCS (inpatient) procedures, each with verbatim evidence,
official-guideline citations and a platform confidence score -> draft claim -> **pre-bill denial check** (MUE units, modifiers, NCCI, code validity on the
date of service, sex/age edits, sequencing...) with the denial code (CARC) it would trigger and how to fix it.

**Before you start**
1. Upload `medical-coding-ai.zip` to **My Drive** (top level). Keep the name, or change `ZIP` below.
2. Runtime -> Change runtime type -> **T4 GPU** (optional; makes the one-time embedding step ~5x faster).
3. Optional LLM: add a Colab secret (key icon on the left) named `OPENROUTER_API_KEY`. Without it the platform runs the deterministic retrieval-only coder.

Use **synthetic or de-identified notes only**. Colab is not a HIPAA environment.

## 1. Mount Drive and unzip the project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP = '/content/drive/MyDrive/medical-coding-ai.zip'      # <- where you put the zip
DRIVE_WORK = '/content/drive/MyDrive/medical-coding-ai-work'  # persistent: vector index, backups, reports

!rm -rf /content/medical-coding-ai && unzip -q -o "$ZIP" -d /content/
!mkdir -p "$DRIVE_WORK" /content/runtime
!ls /content/medical-coding-ai /content/medical-coding-ai/data_sources

## 2. Install (about 1-2 minutes)

In [ ]:
%cd /content/medical-coding-ai/backend
!pip install -q -e ".[kb]" gradio 2>&1 | grep -v -i "already satisfied" | tail -3
!python -c "import sentence_transformers, torch; print('sentence-transformers OK, GPU:', torch.cuda.is_available())"

## 3. Configure (environment variables used by every command below)

In [ ]:
import os, json, secrets, base64
from pathlib import Path

keys_file = Path(DRIVE_WORK) / 'secrets.json'          # generated once, kept on your Drive
if not keys_file.exists():
    keys_file.write_text(json.dumps({
        'JWT_SECRET': secrets.token_urlsafe(48),
        'ENCRYPTION_KEYS': base64.urlsafe_b64encode(os.urandom(32)).decode(),
        'ADMIN_PASSWORD': 'Adm1n-' + secrets.token_urlsafe(9),
    }))
keys = json.loads(keys_file.read_text())

os.environ.update({
    'ENVIRONMENT': 'development',
    'DATABASE_URL': 'sqlite:////content/runtime/medcoding.db',   # fast local disk; rebuilt in ~1 min per session
    'STORAGE_DIR': '/content/runtime/storage',
    'VECTOR_STORE': 'disk',
    'VECTOR_DIR': f'{DRIVE_WORK}/vectors',                      # embeddings persist on Drive (computed once)
    'EMBEDDING_PROVIDER': 'sentence_transformers',               # BAAI/bge-small-en-v1.5 on GPU if available
    'EMBEDDING_MODEL': 'BAAI/bge-small-en-v1.5',
    'ENABLED_CODE_SYSTEMS': 'ICD-10-CM,ICD-10-PCS,HCPCS,CPT',
    'LLM_PROVIDER': 'heuristic',
    'LOG_JSON': 'false', 'LOG_LEVEL': 'WARNING',
    'JWT_SECRET': keys['JWT_SECRET'], 'ENCRYPTION_KEYS': keys['ENCRYPTION_KEYS'],
    'CORS_ORIGINS': '*',
})
try:
    from google.colab import userdata
    k = userdata.get('OPENROUTER_API_KEY')
    if k:
        os.environ.update({'LLM_PROVIDER': 'openrouter', 'OPENROUTER_API_KEY': k,
                           'LLM_MODEL': 'anthropic/claude-sonnet-4.5'})
except Exception:
    pass
print('LLM provider:', os.environ['LLM_PROVIDER'])

## 4. Load the reference data (your CMS/CDC files in `data_sources/`)
ICD-10-CM (+ validity check against the CMS order file), ICD-10-PCS, HCPCS Level II + modifiers, CPT extract, MUE unit limits,
HCPCS processing notes, and the ICD-10-CM / ICD-10-PCS Official Guidelines PDFs.

`--i-have-a-cpt-license`: CPT content is AMA-copyrighted. Keep this flag only if your organization is licensed; otherwise remove it (CPT codes are then not validated).

In [ ]:
!python -m app.cli init-db
!python -m app.cli kb load-all --data-dir /content/medical-coding-ai/data_sources --i-have-a-cpt-license
!python -m app.cli kb status

## 5. Build the semantic index (first run: ~3-5 min on T4 GPU, ~20 min on CPU; later runs reuse Drive)

In [ ]:
!python -m app.cli kb index --system all

## 6. Reference-data freshness (real-world check: FY2026 code sets end 2026-09-30)

In [ ]:
!python -m app.cli kb freshness --dos 2026-09-15
!python -m app.cli kb freshness --dos 2026-10-01

## 7. Tests (optional, ~2 min)

In [ ]:
!python -m pytest -q -p no:warnings 2>&1 | tail -5

## 8. Code sample notes end to end (codes -> evidence -> guidelines -> draft claim -> denial check)

In [ ]:
!python -m app.cli code-note ../data/sample_notes/01_office_visit_knee_injection.txt --sex M --age 67 --dos 2026-09-15

In [ ]:
!python -m app.cli code-note ../data/sample_notes/02_inpatient_lap_chole.txt --encounter inpatient --sex F --age 52 --dos 2026-09-10

In [ ]:
!python -m app.cli code-note ../data/sample_notes/03_urgent_care_laceration.txt --sex F --age 34 --dos 2026-09-20

## 9. Claim scrubber on labeled test claims + your own claim

In [ ]:
!python -m app.evaluation.claims_eval --out "$DRIVE_WORK/claims_eval.json"

In [ ]:
claim = {
  "claim_id": "my-claim",
  "date_of_service": "2026-09-15", "patient_sex": "M", "patient_age": 67,
  "diagnoses": ["M17.11", "E11.9"],
  "lines": [
    {"code": "99214", "units": 1, "charge": 165},
    {"code": "20610", "units": 1, "modifiers": ["RT"], "charge": 150},
    {"code": "J1100", "units": 8, "charge": 2}
  ]
}
Path('/content/runtime/claim.json').write_text(json.dumps(claim))
!python -m app.cli claims scrub /content/runtime/claim.json

## 10. Diagnosis-coding accuracy on the labeled synthetic notes (dev / validation / held-out test)

In [ ]:
for split in ['dev', 'validation', 'test']:
    !python -m app.evaluation.runner --split {split} --baselines retrieval_only 2>&1 | grep -E "^\| (baseline|retrieval)"
# With an OpenRouter key you can also compare LLM baselines:
# !python -m app.evaluation.runner --split test --baselines llm_only,llm_rag,llm_rag_validation,full

## 11. Launch the workbench UI (public *.gradio.live link, valid ~72h)
Tabs: Code a note / Claim scrubber / Code lookup / Official guidelines / Reference data. Stop with the square button.

In [ ]:
!python -m app.demo.gradio_app --share

## 12. (Optional) REST API + Swagger docs
Starts FastAPI in the background and prints a Colab proxy URL for `/docs`. Log in with the admin user created here.

In [ ]:
!python -m app.cli create-user admin@example.org --role admin --password "{keys['ADMIN_PASSWORD']}" || true
!nohup uvicorn app.main:app --host 0.0.0.0 --port 8000 > /content/runtime/api.log 2>&1 &
import time; time.sleep(8)
from google.colab.output import eval_js
print('Swagger UI:', eval_js("google.colab.kernel.proxyPort(8000)") + 'docs')
print('Login: admin@example.org /', keys['ADMIN_PASSWORD'])

## 13. Save a backup of the database to Drive (optional)

In [ ]:
!cp /content/runtime/medcoding.db "$DRIVE_WORK/medcoding-$(date +%Y%m%d-%H%M).db" && ls -la "$DRIVE_WORK"